# 02 · Step-by-step exploration: NER, then targeted tone

**Scope.** German only, one block-A vote (**A-631**, Begrenzungsinitiative,
2020-09-27), every person and organisation, tone *toward* the actor only.
Each step's output is read before the next one starts.

## 1. The slice

The archive for A-631 holds every article from the 34 outlets published inside
the event window. The query filtered on outlet, dates and language, **not on
topic**. This step keeps the German rows and checks them against the pull
receipt. Nothing here prints article text, so the notebook can be committed
with its outputs.

In [9]:
import html
import json
import re

import pandas as pd

from mediapos import config as cfg, queries as q

EVENT_ID = "A-631"
LANGUAGE = "de"

query = pd.read_csv(cfg.DATA_OUT / "queries.csv").set_index("event_id").loc[EVENT_ID]
receipts = json.loads((cfg.DATA_OUT / "pull_receipts.json").read_text())
receipt = receipts[f"{query.date_from}_{query.date_to}"]
archive = cfg.REPO_ROOT / receipt["local_path"]

print(f"event    {EVENT_ID}  window {query.date_from} → {query.date_to}")
print(f"archive  {archive.relative_to(cfg.REPO_ROOT)}")
print(f"receipt  {receipt['articles_read']:,} articles, {receipt['by_language']}")

event    A-631  window 2020-08-28 → 2020-10-11
archive  data/corpus/2020-08-28_2020-10-11.tsv.xz
receipt  87,403 articles, {'de': 58649, 'fr': 28754}


In [10]:
articles = pd.DataFrame(r for r in q.read_articles(archive) if r["language"] == LANGUAGE)
articles["pubtime"] = pd.to_datetime(articles["pubtime"], utc=True)
articles["char_count"] = articles["char_count"].astype(int)

# The rows must be exactly what the receipt says was downloaded.
assert len(articles) == receipt["by_language"][LANGUAGE], len(articles)

print(f"{len(articles):,} {LANGUAGE} rows, count matches the receipt")
print(f"{articles['id'].nunique():,} distinct article ids")
print(f"published {articles.pubtime.min():%Y-%m-%d} → {articles.pubtime.max():%Y-%m-%d}")

58,649 de rows, count matches the receipt
58,587 distinct article ids
published 2020-08-27 → 2020-10-11


### One id, several rows: revisions of the same article

Some ids appear two or three times. These rows have the same outlet and
timestamp but a different `content_id` and body: they are **edited versions**
of one online article. Counting every version would weight an article by how
often it was edited, so we keep **one row per id: the longest version**, with
ties broken by `content_id` so the choice is deterministic.

In [11]:
versions = articles.groupby("id").size()
revised = articles[articles["id"].isin(versions[versions > 1].index)]
print(f"{(versions > 1).sum()} ids have several versions ({len(revised)} rows)")
print(revised["medium_code"].value_counts().to_string())

articles = (articles.sort_values(["id", "char_count", "content_id"], ascending=[True, False, True])
                    .drop_duplicates("id").reset_index(drop=True))
assert articles["id"].is_unique
print(f"\nslice: {len(articles):,} articles, one per id")

58 ids have several versions (120 rows)
medium_code
BLIO    106
ZWAO      6
NNTA      4
NNBU      2
SRF       2

slice: 58,587 articles, one per id


### Who published them, and in what form

In [12]:
(articles.groupby(["medium_code", "medium_name"]).size()
         .rename("articles").sort_values(ascending=False).reset_index())

,medium_code,medium_name,articles
0,AZM,Aargauer Zeitung / MLZ,4567
1,SGTO,tagblatt.ch,4392
2,BLIO,blick.ch,4239
3,NNBE,bernerzeitung.ch,4007
4,SRF,srf.ch,3601
5,NNBS,bazonline.ch,3175
6,NNTA,tagesanzeiger.ch,3149
7,ZWAO,20 minuten online,3145
8,NNBU,derbund.ch,3053
9,LUZ,Luzerner Zeitung,2881


In [13]:
articles["doctype_description"].value_counts().rename("articles").to_frame()

,articles
doctype_description,
Online medium,31540
Regional daily newspaper,14450
National daily newspaper,10767
Regional weekly newspaper,1830


In [14]:
articles["char_count"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_frame()

,char_count
count,58587.0
mean,4602.0
std,10304.0
min,86.0
10%,896.0
25%,1741.0
50%,3288.0
75%,5120.0
90%,7570.0
max,551056.0


## 2. The articles about the vote

The window holds everything the outlets published, so we keep the articles that
**name the initiative**: its common name (*Begrenzungsinitiative*), its
opponents' name (*Kündigungsinitiative*) or its official title (*massvolle
Zuwanderung*). We search the headline, subhead and body.

*Personenfreizügigkeit* is left out on purpose. It also appears in articles
about the EU framework agreement, so it would pull in more than this vote.

Naming the initiative once does not make an article *about* it. A portrait of
the cantonal CVP can mention it in passing. So we also count the mentions and
record whether the name is in the headline, which shows how central the vote
is to each article.

In [15]:
VOTE = re.compile(
    r"Begrenzungs-?initiative|Kündigungs-?initiative|massvollen? Zuwanderung", re.IGNORECASE
)

searchable = articles["head"] + "\n" + articles["subhead"] + "\n" + articles["content"]
# pandas' str.count drops a compiled pattern's IGNORECASE flag, so count with findall.
n_mentions = searchable.map(lambda s: len(VOTE.findall(s)))
vote = articles[n_mentions > 0].assign(n_vote_mentions=n_mentions[n_mentions > 0]).reset_index(drop=True)
vote["vote_in_head"] = (vote["head"] + " " + vote["subhead"]).str.contains(VOTE)

print(f"{len(vote):,} of {len(articles):,} articles name the initiative "
      f"({100 * len(vote) / len(articles):.1f}%)")
print(f"{vote['vote_in_head'].sum()} name it in the headline or subhead")
vote["n_vote_mentions"].clip(upper=5).value_counts().sort_index().rename(
    lambda n: "5+" if n == 5 else n).rename("articles").to_frame()

1,086 of 58,587 articles name the initiative (1.9%)
82 name it in the headline or subhead


,articles
n_vote_mentions,
1,544
2,182
3,127
4,73
5+,160


In [16]:
(vote.groupby("medium_code")
     .agg(articles=("id", "size"), in_head=("vote_in_head", "sum"),
          median_mentions=("n_vote_mentions", "median"))
     .join(articles.groupby("medium_code").size().rename("all_articles"))
     .assign(share_pct=lambda t: (100 * t.articles / t.all_articles).round(1))
     .sort_values("articles", ascending=False))

,articles,in_head,median_mentions,all_articles,share_pct
medium_code,,,,,
SGTO,218,9,3.0,4392,5.0
AZM,79,0,1.0,4567,1.7
SRF,64,12,3.0,3601,1.8
NNBE,60,12,1.5,4007,1.5
NNBS,57,12,2.0,3175,1.8
SGT,53,0,1.0,2794,1.9
NNBU,53,12,2.0,3053,1.7
NNTA,50,11,2.0,3149,1.6
LUZ,48,1,1.0,2881,1.7


## 3. From markup to text

`content` is not plain text. It is XML-like markup, and each tag needs a decision:

| Tag | What it is | Decision |
|---|---|---|
| `<ld>` | lead paragraph | **body** |
| `<p>` | paragraph | **body** |
| `<zt>` | crosshead (subheading) | **body**, on its own line |
| `<a>`, `<br>`, tables | links, line breaks, cells | **body**, the markup removed and the text kept |
| `<lg>` | photo caption | own column `captions` |
| `<ka>` | box, often a pull quote that repeats the body | own column `boxes` |
| `<nt>` | note, e.g. a line about the author | own column `notes` |
| `<au>` | author byline | **dropped** |
| `<ur>` | photo credit | **dropped** |

Bylines and photo credits are dropped because NER would read the journalist's
or photographer's name as an actor in the story. Captions, boxes and notes are
kept apart rather than thrown away. Captions are visibility too (a photo of an
actor), and pull quotes would count the same mention twice if merged into the
body. Whether to use them is a later decision.

The model input `text` is headline, subhead and body, separated by blank
lines.

In [17]:
SIDE_COLUMNS = {"lg": "captions", "ka": "boxes", "nt": "notes"}
DROPPED = ("au", "ur")
LINE_BREAK = re.compile(r"</?(?:p|zt|ld|tx|br|tr|table)\b[^>]*>|</t[dh]>")
ANY_TAG = re.compile(r"<[^>]+>")


def element(tag: str) -> re.Pattern[str]:
    return re.compile(rf"<{tag}\b[^>]*>(.*?)</{tag}>", re.DOTALL)


def to_text(markup: str) -> str:
    """Markup fragment to plain text: one line per block, whitespace collapsed."""
    # Inline tags become a space: "einen<a ...>Bausatz" must not fuse into one word.
    text = html.unescape(ANY_TAG.sub(" ", LINE_BREAK.sub("\n", markup)))
    lines = (re.sub(r"\s+", " ", line).strip() for line in text.split("\n"))
    return "\n".join(line for line in lines if line)


def split_content(markup: str) -> dict[str, str]:
    # Drop first: photo credits also sit inside captions.
    for tag in DROPPED:
        markup = element(tag).sub(" ", markup)
    parts = {}
    for tag, column in SIDE_COLUMNS.items():
        parts[column] = "\n".join(to_text(m) for m in element(tag).findall(markup))
        markup = element(tag).sub("\n", markup)
    parts["body"] = to_text(markup)
    return parts


vote = vote.join(pd.DataFrame(vote["content"].map(split_content).tolist()))
vote["text"] = [
    "\n\n".join(p for p in (h.strip(), s.strip(), b) if p)
    for h, s, b in zip(vote["head"], vote["subhead"], vote["body"])
]

### Checks

The cleaning is only trusted if nothing of the markup survives and nothing of
the body is lost on the way.

In [18]:
leftover_tag = vote["text"].str.contains(r"</?[a-z]+\b[^>]*>")
leftover_entity = vote["text"].str.contains(r"&[a-z]+;|&#\d+;")
assert not leftover_tag.any(), vote.loc[leftover_tag, "id"].tolist()
assert not leftover_entity.any(), vote.loc[leftover_entity, "id"].tolist()
assert (vote["body"].str.len() > 0).all()

# Every word of the raw markup must land somewhere: body, a side column, or a
# dropped tag. Compare word counts, not strings, since whitespace changed.
words = lambda s: len(re.findall(r"\w+", s))
dropped = vote["content"].map(lambda m: sum(words(x) for t in DROPPED for x in element(t).findall(m)))
kept = vote[["body", *SIDE_COLUMNS.values()]].map(words).sum(axis=1)
raw = vote["content"].map(lambda m: words(html.unescape(ANY_TAG.sub(" ", m))))
assert (kept + dropped == raw).all(), (kept + dropped - raw).abs().max()

print("no tag or HTML entity left; every word accounted for")
print(f"articles with captions {(vote.captions != '').sum()}, boxes {(vote.boxes != '').sum()}, "
      f"notes {(vote.notes != '').sum()}")
print(f"words: body {vote.body.map(words).sum():,}, captions {vote.captions.map(words).sum():,}, "
      f"boxes {vote.boxes.map(words).sum():,}, notes {vote.notes.map(words).sum():,}, "
      f"dropped {dropped.sum():,}")

no tag or HTML entity left; every word accounted for
articles with captions 721, boxes 340, notes 25
words: body 2,829,365, captions 260,528, boxes 55,690, notes 543, dropped 2,170


In [19]:
vote["text_chars"] = vote["text"].str.len()
vote["text_chars"].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(0).to_frame()

,text_chars
count,1086.0
mean,17974.0
std,32595.0
min,381.0
10%,1638.0
25%,3119.0
50%,4805.0
75%,9148.0
90%,73211.0
99%,136561.0


### Reading one article

Run the cell below to read an article in full, before and after cleaning. **Clear
its output before committing.** Article text is licensed corpus content and
must not enter git.

In [ ]:
def show(i: int) -> None:
    row = vote.iloc[i]
    print(f"{row['id']}  {row['medium_name']}  {row['pubtime']:%Y-%m-%d}  "
          f"mentions={row['n_vote_mentions']}  in_head={row['vote_in_head']}\n")
    print(row["text"])
    for column in SIDE_COLUMNS.values():
        if row[column]:
            print(f"\n--- {column} ---\n{row[column]}")

import random
article_id = random.randint(0, len(vote))
show(article_id)